# Image Data Preparation and Preprocessing for Computer Vision

**Course 13 – Computer Vision Engineer**

This notebook follows the journey from an image file to a dataset that is ready for a CNN.

We will work through:
1. How images are represented as numbers
2. Image dimensions, channels, data types and pixel ranges
3. Image loading and inspection
4. Resizing and normalization
5. Image augmentation
6. Training, validation and test splits
7. Class balance and data leakage
8. A complete, reusable image pipeline
9. How to adapt the same thinking to a malaria-cell image project

**Main practical dataset:** CIFAR-10. It is small, familiar and can be downloaded directly through `torchvision`. We use it to demonstrate the pipeline. Later, the same ideas can be applied to a folder-based dataset such as malaria-infected and uninfected cell images.

> Important: The purpose today is to learn and inspect the data pipeline. We are not trying to build the best possible classifier in this notebook.

## 1. Before writing code: how should we think about an image project?

A computer does not receive the meaning of a photograph. It receives numerical values.

For every image project, ask these questions before training:
- What does one image represent?
- What are the classes or labels?
- Are images colour or grayscale?
- Are image sizes consistent?
- What pixel range and data type are being used?
- What changes are realistic in the real world?
- Could two images be related, for example, from the same patient, slide, video or original photograph?
- How will we create a fair test set?

The answers guide the preprocessing choices. There is no single preprocessing recipe that is correct for every image problem.

## 2. Packages used in this notebook

We will use a small set of common Python packages.

| Package | Why we use it |
|---|---|
| `torch` | Main PyTorch library. It works with tensors and provides tools used to build and train deep-learning models. |
| `torchvision` | Image-related tools for PyTorch. It provides datasets, image transformations and utilities. |
| `numpy` | Helps us inspect and work with numerical arrays. |
| `matplotlib` | Displays images and plots so we can visually check our data. |
| `PIL` (Pillow) | Opens and handles common image files such as JPG and PNG. |
| `collections` | Python's built-in tools for counting labels and checking class balance. |
| `pathlib` | Handles folder and file paths in a clean, cross-platform way. |

If a package is not installed in your environment, install it in a terminal or notebook cell. In a classroom environment, it is better to install packages before the session.

In [ ]:
# Install only if a package is missing.
# In Jupyter, run this cell when needed.

# %pip install torch torchvision numpy matplotlib pillow

In [ ]:
# Import the packages we will use in the notebook.

import torch                         # Tensors and PyTorch functions
import torchvision                   # Image datasets and image transformations
from torchvision import transforms   # Tools for resizing, converting and augmenting images
from torchvision.datasets import CIFAR10  # A small, labelled image dataset
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np                   # Numerical arrays and calculations
import matplotlib.pyplot as plt      # Display images and charts
from PIL import Image                # Open and inspect image files
from collections import Counter      # Count how many images belong to each class
from pathlib import Path             # Work with folder paths

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

## 3. What is an image made of?

A digital image is a grid of small units called **pixels**.

- A pixel is one small picture element.
- In a grayscale image, a pixel commonly stores one intensity value.
- In an 8-bit grayscale image, values usually range from 0 to 255.
- `0` represents black, `255` represents white, and values between them represent shades of gray.
- In an RGB image, each pixel has three values: Red, Green and Blue.

For example, a colour pixel might be:

`[255, 0, 0]` → red  
`[0, 255, 0]` → green  
`[0, 0, 255]` → blue

A computer stores these values in arrays or tensors.

In [ ]:
# Create a tiny grayscale image with 4 rows and 5 columns.
# Each number represents the brightness of one pixel.

tiny_gray_image = np.array([
    [0,  40,  80, 120, 160],
    [20, 60, 100, 140, 180],
    [40, 80, 120, 160, 200],
    [60, 100, 140, 180, 255]
], dtype=np.uint8)

print("Image values:")
print(tiny_gray_image)
print("Image shape:", tiny_gray_image.shape)  # (height, width)
print("Minimum pixel value:", tiny_gray_image.min())
print("Maximum pixel value:", tiny_gray_image.max())

plt.figure(figsize=(5, 3))
plt.imshow(tiny_gray_image, cmap="gray", vmin=0, vmax=255)
plt.colorbar(label="Pixel intensity")
plt.title("A small grayscale image")
plt.axis("off")
plt.show()

### What does image shape mean?

For a grayscale image, shape is commonly:

`(height, width)`

For a colour image in a channels-last layout, shape is commonly:

`(height, width, channels)`

For example, an RGB image of 100 pixels high and 150 pixels wide has shape:

`(100, 150, 3)`

PyTorch commonly uses a channels-first layout:

`(channels, height, width)`

So the same image may have shape `(3, 100, 150)` after conversion to a PyTorch tensor.

**Why does this matter?** A model expects a particular input shape. If we pass the wrong shape or channel order, the model may fail or interpret the image incorrectly.

## 4. Load a real image dataset

We will use CIFAR-10. It contains 60,000 small colour images in 10 categories. Each image is 32 × 32 pixels.

The dataset is useful for learning because:
- It is widely used for introductory image-classification practice.
- `torchvision` can download it for us.
- It already includes class labels.
- It is small enough for classroom demonstrations.

**Alternative:** For a custom project, images may be stored in folders, such as `cats/`, `dogs/`, `infected/` and `uninfected/`. Later in this notebook we will look at how to load that type of dataset.

In [ ]:
# CIFAR-10 will be downloaded to this folder the first time.
# Set download=True only when you have an internet connection.

data_folder = Path("data")

# To inspect images in their original PIL format, we first use a simple transform.
# We will apply resizing and normalization later.
basic_transform = transforms.ToTensor()

cifar_dataset = CIFAR10(
    root=str(data_folder),
    train=True,
    download=True,
    transform=basic_transform
)

print("Number of training images:", len(cifar_dataset))
print("Class names:", cifar_dataset.classes)

### What is a dataset object?

A dataset object connects an image to its label.

When we request one item, it gives us:
- the image
- the correct class label

For CIFAR-10, the label is a number from 0 to 9. The class-name list tells us what each number means.

A dataset does not necessarily load every image into memory at once. Many dataset classes load an item when it is requested.

In [ ]:
# Read one image and its label from the dataset.

image_tensor, label_number = cifar_dataset[0]

print("Image tensor shape:", image_tensor.shape)
print("Label number:", label_number)
print("Label name:", cifar_dataset.classes[label_number])
print("Tensor data type:", image_tensor.dtype)
print("Pixel minimum:", image_tensor.min().item())
print("Pixel maximum:", image_tensor.max().item())

## 5. Display images before changing them

Always look at the data before preprocessing.

A quick visual inspection can reveal:
- incorrect labels
- blank or corrupted images
- unexpected colour channels
- objects cut off at the edge
- duplicates or near-duplicates
- class imbalance

Do not assume that a dataset is clean just because it downloaded successfully.

In [ ]:
# Display a small sample of images with their labels.

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for index, ax in enumerate(axes.flat):
    image, label = cifar_dataset[index]

    # ToTensor converts an image to channels-first format.
    # Matplotlib expects channels-last format, so we move the channel dimension.
    image_for_display = image.permute(1, 2, 0).numpy()

    ax.imshow(image_for_display)
    ax.set_title(cifar_dataset.classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Image dimensions and resolution

Resolution describes how many pixels an image contains.

For example:
- 32 × 32 = 1,024 pixels
- 224 × 224 = 50,176 pixels
- 640 × 480 = 307,200 pixels

If width and height are both doubled, the number of pixels becomes four times larger.

### Why resize images?

A neural network usually processes images in batches. Images in one batch need compatible dimensions.

Resizing can:
- make image dimensions consistent
- reduce memory use when shrinking large images
- match the input size expected by a pretrained model

### What are the alternatives?

| Method | When it can be useful | What to watch |
|---|---|---|
| Direct resize | When a fixed shape is required and distortion is acceptable | May stretch objects |
| Resize while preserving aspect ratio | When object proportions matter | Resulting dimensions may differ |
| Resize and pad | When the full image must remain visible | Adds empty or artificial borders |
| Crop | When the target region is important | May remove useful context |

For a pretrained model, use the image size and resize method recommended by that model.

In [ ]:
# Resize one CIFAR-10 image to 224 x 224 for demonstration.
# CIFAR-10 images are 32 x 32, so this enlarges them.
# Enlarging does not create new image detail; it only changes the dimensions.

original_pil, original_label = CIFAR10(
    root=str(data_folder), train=True, download=True
)[0]

resize_transform = transforms.Resize((224, 224))
resized_pil = resize_transform(original_pil)

print("Original size:", original_pil.size)  # PIL reports (width, height)
print("Resized size:", resized_pil.size)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(original_pil)
axes[0].set_title("Original: 32 x 32")
axes[0].axis("off")
axes[1].imshow(resized_pil)
axes[1].set_title("Resized: 224 x 224")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Pixel scaling and normalization

Images stored as 8-bit values commonly have pixel values from 0 to 255.

A common first step is to divide every pixel by 255:

`new_value = old_value / 255`

Examples:
- 0 becomes 0.000
- 128 becomes approximately 0.502
- 255 becomes 1.000

This puts the values in the range 0 to 1.

### Why do this?

Neural networks often train more comfortably when inputs are on a sensible numerical scale. Scaling also makes the input range predictable.

### Is dividing by 255 always correct?

No. It is a common choice, not a universal rule.

Some pretrained models expect **channel-wise standardization**:

`standardized_value = (pixel_value - channel_mean) / channel_std`

Here:
- `pixel_value` is the scaled channel value
- `channel_mean` is the expected average for that channel
- `channel_std` is the expected standard deviation for that channel

Use the preprocessing specified by the model's documentation. If a pretrained model expects a particular mean and standard deviation, do not replace it with a different normalization without a reason.

In [ ]:
# Demonstrate scaling with a few example pixel values.

pixel_values = torch.tensor([0, 128, 255], dtype=torch.float32)

# Convert values from the range 0-255 to the range 0-1.
scaled_values = pixel_values / 255.0

print("Original values:", pixel_values)
print("Scaled values:", scaled_values)

## 8. Convert an image into a PyTorch tensor

`transforms.ToTensor()` performs two useful jobs for common PIL images:
1. Converts the image into a PyTorch tensor.
2. Changes the layout from height-width-channel to channel-height-width.
3. For standard 8-bit PIL images, it also scales values from 0–255 to 0–1.

That is why we should inspect the tensor after conversion instead of assuming its shape and range.

In [ ]:
# Convert a PIL image into a PyTorch tensor.

to_tensor = transforms.ToTensor()
example_tensor = to_tensor(original_pil)

print("PIL image size (width, height):", original_pil.size)
print("Tensor shape (channels, height, width):", example_tensor.shape)
print("Tensor data type:", example_tensor.dtype)
print("Minimum:", example_tensor.min().item())
print("Maximum:", example_tensor.max().item())

## 9. Other preprocessing choices

Preprocessing should solve a problem in the data or meet the model's input requirements.

| Operation | Why it may be used | When to be careful |
|---|---|---|
| RGB to grayscale | When colour is not important or a one-channel input is expected | Colour may carry useful class information |
| Denoising | To reduce unwanted sensor noise | Strong denoising can remove fine details |
| Contrast adjustment | To improve visibility in low-contrast images | Can change the appearance of meaningful features |
| Cropping | To focus on a region of interest | May remove context or the object |
| Data type conversion | To use a numeric type expected by the model, often `float32` | Check memory and expected range |

For medical images, microscopy, satellite images and industrial inspection, preprocessing choices should be based on the image source and the task. Do not apply an operation just because it is common in another project.

## 10. Data augmentation: show the model realistic variation

Data augmentation creates modified versions of training images.

Imagine a model that sees only centred objects in bright lighting. It may struggle when the same kind of object appears slightly rotated, shifted or under different lighting.

Augmentation can help the model learn patterns that remain useful across these changes.

Common transformations:
- horizontal flip
- small rotation
- random crop
- slight zoom
- brightness and contrast changes
- mild blur or noise

**Important:** The transformation must preserve the label. A horizontal flip may be fine for many everyday object classes, but it can be wrong for text recognition or tasks where left and right have different meanings.

In [ ]:
# Create a few augmented versions of one image.
# These transforms are for demonstration; the random result may change each time.

augmentation_demo = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),  # Always flip for this demonstration
    transforms.RandomRotation(degrees=15)     # Rotate by a random angle within +/- 15 degrees
])

flipped_rotated_image = augmentation_demo(original_pil)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(original_pil)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(flipped_rotated_image)
axes[1].set_title("Flipped and rotated")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### A more complete augmentation example

The following transformations are often used for natural-image classification:
- random horizontal flip
- small random rotation
- random resized crop
- mild colour variation

The values below are examples, not rules. Start with mild changes and inspect the results.

In [ ]:
# Build an augmentation pipeline for training images.

train_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor()
])

# Create several versions of the same original image.
fig, axes = plt.subplots(1, 4, figsize=(11, 3))

for ax in axes:
    augmented = train_augmentation(original_pil)
    ax.imshow(augmented.permute(1, 2, 0).numpy())
    ax.axis("off")

axes[0].set_title("Variation 1")
axes[1].set_title("Variation 2")
axes[2].set_title("Variation 3")
axes[3].set_title("Variation 4")
plt.suptitle("Different augmented views of one training image")
plt.tight_layout()
plt.show()

### When should we not use a transformation?

| Task | Possible issue |
|---|---|
| OCR / document reading | Flipping makes text unreadable |
| Medical cell images | Some rotations or colour changes may alter meaningful patterns |
| Road signs | Flipping can change the meaning of a sign |
| Digit recognition | Rotation may turn one digit into another |
| Face or pose tasks | Cropping or flipping may affect orientation-related labels |

For object detection, the bounding boxes must be transformed along with the image. For segmentation, the mask must be transformed in exactly the same geometric way as the image. If only the image changes, the labels become incorrect.

## 11. Training, validation and test datasets

We do not use every image for the same purpose.

- **Training set:** The model learns its weights from these images.
- **Validation set:** We use it during development to compare models and tune settings.
- **Test set:** We keep it separate for a final evaluation after decisions are complete.

A common starting split is 70% training, 15% validation and 15% testing. Other choices, such as 80/10/10, can also be appropriate.

The best split depends on:
- how many images are available
- how many classes exist
- whether classes are balanced
- whether images are related to one another
- whether the data represents different people, sessions, locations or time periods

## 12. Why stratification matters

Suppose a dataset contains:
- 600 cat images
- 400 dog images

That means 60% cats and 40% dogs.

A plain random split may accidentally create a training set with a very different class ratio. **Stratified splitting** aims to preserve the class proportions in each subset.

Stratification is helpful for classification when class proportions matter. It does not solve every problem: it does not remove duplicates, fix incorrect labels or prevent leakage between related images.

In [ ]:
# Show the class distribution in CIFAR-10.
# The dataset labels are integers from 0 to 9.

label_counts = Counter(cifar_dataset.targets)

for class_id, class_name in enumerate(cifar_dataset.classes):
    print(class_name, ":", label_counts[class_id])

## 13. Make train, validation and test splits

CIFAR-10 already provides a standard training portion and a standard test portion.

For this classroom exercise:
- We will split the provided training portion into training and validation.
- We will keep the official test portion separate.
- We will use a fixed random seed so the split can be reproduced.

For a custom dataset, do not assume that a random split is always safe. If several images come from the same patient, slide, video, object or original photograph, split by that group so related images do not appear across training and testing.

In [ ]:
# Load CIFAR-10 training and test datasets.
# We first load them without transformations so we can attach different
# transformations to the training and validation subsets later.

raw_train_dataset = CIFAR10(
    root=str(data_folder),
    train=True,
    download=True,
    transform=None
)

raw_test_dataset = CIFAR10(
    root=str(data_folder),
    train=False,
    download=True,
    transform=None
)

# Keep 15% of the original training portion for validation.
validation_size = int(0.15 * len(raw_train_dataset))
training_size = len(raw_train_dataset) - validation_size

# A fixed generator makes this random split repeatable.
generator = torch.Generator().manual_seed(42)

train_subset, validation_subset = random_split(
    raw_train_dataset,
    [training_size, validation_size],
    generator=generator
)

print("Training images:", len(train_subset))
print("Validation images:", len(validation_subset))
print("Test images:", len(raw_test_dataset))

### A note about stratified splitting

The simple `random_split` above is reproducible, but it does not guarantee the same class proportions in each subset.

For a classification dataset where class balance is important, use a stratified split. One common option is `train_test_split` from scikit-learn with `stratify=labels`.

For datasets with related images, group-aware splitting is often more important than ordinary stratification. For example, keep all images from one patient in only one subset.

We will use scikit-learn for a short example of stratification.

In [ ]:
# Install scikit-learn if it is not already installed.
# %pip install scikit-learn

from sklearn.model_selection import train_test_split

# CIFAR-10 labels are available as a list of integers.
all_indices = np.arange(len(raw_train_dataset))
all_labels = np.array(raw_train_dataset.targets)

# First separate 15% for the test/holdout example.
# stratify=all_labels aims to preserve class proportions.
train_indices, holdout_indices = train_test_split(
    all_indices,
    test_size=0.15,
    random_state=42,
    stratify=all_labels
)

# Split the remaining data so validation is approximately 15% of the full dataset.
# 0.15 / 0.85 gives the required fraction of the remaining data.
train_labels = all_labels[train_indices]
train_indices, validation_indices = train_test_split(
    train_indices,
    test_size=(0.15 / 0.85),
    random_state=42,
    stratify=train_labels
)

print("Training:", len(train_indices))
print("Validation:", len(validation_indices))
print("Holdout:", len(holdout_indices))

## 14. Data leakage: a very important warning

Data leakage occurs when information that should be unavailable during training or model selection influences the result.

A common image-project mistake is splitting individual images when several images are actually related.

Examples:
- different frames from the same video
- multiple photographs of the same patient
- several microscope images from one slide
- repeated images of the same product
- original images and near-identical edited copies

If related images appear in both training and test sets, the model may appear to perform very well because it has effectively seen very similar examples before.

**Better approach:** Identify the group first, then split whole groups. In medical imaging, patient-level splitting is often essential.

## 15. Build a clean pipeline with different transforms for each split

The training set receives random augmentation because we want the model to see different realistic versions of training examples.

Validation and test sets should receive consistent preprocessing only. We do not randomly change them each time because we want stable and fair evaluation.

For this CIFAR-10 demonstration, the images are already 32 × 32. We keep that size so we do not unnecessarily enlarge the dataset.

In [ ]:
# Training pipeline: random augmentation plus tensor conversion.
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

# Validation and test pipeline: no random augmentation.
evaluation_transform = transforms.Compose([
    transforms.ToTensor()
])

# A small wrapper applies the correct transform to a subset of a dataset.
class DatasetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]

        # Apply the selected transform to the image.
        if self.transform:
            image = self.transform(image)

        return image, label

# Attach training augmentation only to the training subset.
train_dataset = DatasetWithTransform(train_subset, train_transform)

# Validation data gets only the stable evaluation transform.
validation_dataset = DatasetWithTransform(validation_subset, evaluation_transform)

# Test data also gets only the stable evaluation transform.
test_dataset = DatasetWithTransform(
    torch.utils.data.Subset(raw_test_dataset, range(len(raw_test_dataset))),
    evaluation_transform
)

print("Pipeline datasets are ready.")

## 16. What is a DataLoader?

A `DataLoader` takes items from a dataset and groups them into batches.

Why use batches?
- It is more efficient than processing one image at a time.
- It helps use the GPU efficiently.
- It lets us shuffle training examples between epochs.
- It makes it easier to control memory use.

A common starting batch size is 32 or 64. Larger batches need more memory. The right value depends on image size, model size and available hardware.

In [ ]:
# Create data loaders.
# shuffle=True is used for training so the model does not see examples
# in the exact same order every epoch.
# For validation and testing, shuffle is usually not needed.

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Inspect one batch.
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Batch image data type:", images.dtype)

### Read the batch shape

For CIFAR-10, a batch of 64 images commonly has shape:

`(64, 3, 32, 32)`

Read it from left to right:
- 64 = number of images in this batch
- 3 = RGB channels
- 32 = image height
- 32 = image width

This is the channels-first format used by PyTorch.

In [ ]:
# Display a few images from one training batch.

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i, ax in enumerate(axes):
    image = images[i].permute(1, 2, 0).numpy()
    ax.imshow(image)
    ax.set_title(raw_train_dataset.classes[labels[i].item()])
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. A folder-based dataset: how this changes for a real project

Many custom image datasets are arranged like this:

```text
image_dataset/
    train/
        infected/
        uninfected/
    validation/
        infected/
        uninfected/
    test/
        infected/
        uninfected/
```

PyTorch's `ImageFolder` can read this structure. Each subfolder name becomes a class label.

For a real medical dataset, make sure the split was created at the correct group level, such as patient or slide, before placing images into these folders. Do not create a random image-level split if images from the same patient can appear in more than one folder.

In [ ]:
# Example folder-loading code for a custom dataset.
# Change the path to the location of your own image folders.
# This cell is a template and will work after the folders exist.

# from torchvision.datasets import ImageFolder
#
# custom_train = ImageFolder(
#     root="image_dataset/train",
#     transform=train_transform
# )
#
# custom_validation = ImageFolder(
#     root="image_dataset/validation",
#     transform=evaluation_transform
# )
#
# custom_test = ImageFolder(
#     root="image_dataset/test",
#     transform=evaluation_transform
# )
#
# print("Class names:", custom_train.classes)
# print("Training image count:", len(custom_train))

## 18. How to think about the upcoming malaria-cell image task

For the next assignment, students will prepare a malaria-cell image dataset. The aim is to build a reliable preprocessing pipeline, not to make medical decisions.

A possible folder structure is:

```text
malaria_dataset/
    Parasitized/
    Uninfected/
```

Before splitting, students should investigate the dataset documentation:
- What does each image represent?
- Are labels `Parasitized` and `Uninfected`?
- Are multiple images from the same patient, slide or source?
- Are there duplicates?
- Are image sizes and colour channels consistent?
- Are the classes balanced?
- Are there acquisition differences between groups?

### A sensible starting workflow

1. Inspect the dataset and label names.
2. Count images in each class.
3. Display examples from every class.
4. Check image dimensions, channels and pixel ranges.
5. Identify patient/slide/source identifiers if available.
6. Split by patient or slide when those identifiers exist.
7. Use stratification where appropriate.
8. Resize only if required by the selected model.
9. Apply the model-appropriate scaling or normalization.
10. Apply carefully chosen augmentation to training images only.
11. Inspect the final batches and save the split information.

**Medical-image caution:** Do not assume that every natural-image augmentation is suitable for microscopy images. Preserve diagnostic details and follow the dataset's usage guidance. This exercise is for learning image-data engineering, not clinical diagnosis.

## 19. Optional: inspect a custom image file

When students receive their own dataset, they can use this small inspection function to look at one image.

Change the file path to an image that exists on your computer.

In [ ]:
# Example: inspect one image file from a custom dataset.
# Replace the path below with a real JPG or PNG file.

# image_path = Path("malaria_dataset/Parasitized/example.jpg")
#
# if image_path.exists():
#     image = Image.open(image_path)
#     print("Image format:", image.format)
#     print("Image size:", image.size)
#     print("Colour mode:", image.mode)
#     display(image)
# else:
#     print("Please update image_path to point to an existing image.")

## 20. Common mistakes and how to avoid them

| Mistake | Why it is a problem | Better practice |
|---|---|---|
| Resizing without checking aspect ratio | Objects may become distorted | Choose resize, pad or crop based on the task |
| Applying random augmentation to test data | Evaluation becomes inconsistent | Keep evaluation transforms deterministic |
| Using the wrong normalization | Input distribution may not match model expectations | Follow the model's preprocessing requirements |
| Splitting related images across subsets | Test performance can be misleading | Split by patient, slide, video or source group |
| Ignoring class counts | A rare class may be poorly represented | Inspect class distribution and use stratification where suitable |
| Applying the same augmentation to every task | Some changes can alter label meaning | Check whether the transformation preserves the label |
| Trusting the dataset without inspection | Labels, files or images may be incorrect | Visualize samples and validate the data |

A good pipeline is not the one with the most transformations. It is the one where every step has a clear reason.

## 21. Short classroom exercises

Try these before moving to a CNN training project.

**Exercise 1 – Read image shape**
- What does `(3, 32, 32)` represent?
- How would the shape change for a batch of 16 images?

**Exercise 2 – Pixel scaling**
- What does a pixel value of 200 become after dividing by 255?
- Why do we not simply subtract 255 from every pixel?

**Exercise 3 – Choose augmentation**
- Would horizontal flipping be suitable for a street-sign classifier?
- Would it be suitable for a cat-versus-dog classifier?
- Explain your reasoning.

**Exercise 4 – Dataset splitting**
- You have 1,000 images from 20 patients.
- Each patient has 50 images.
- Why could a random image-level split be unsafe?
- What would a patient-level split look like?

**Exercise 5 – Preprocessing decision**
- A pretrained model expects 224 × 224 RGB images and a specific mean and standard deviation.
- Which steps should you apply before inference?
- Where would you find the exact expected values?

## 22. Final checklist: before training any image model

Use this checklist at the start of every image project.

- [ ] I understand what one image represents.
- [ ] I know what each label means.
- [ ] I have inspected examples from every class.
- [ ] I have checked image dimensions, channels and data types.
- [ ] I know the pixel value range.
- [ ] I have selected resizing and normalization based on the model.
- [ ] I have checked class balance.
- [ ] I have considered duplicates and related images.
- [ ] I have created training, validation and test splits correctly.
- [ ] I have applied augmentation only where it is appropriate.
- [ ] I have checked that transformations preserve labels.
- [ ] I can reproduce the same split and preprocessing steps.

**The key idea:** A CNN can only learn from the data it receives. A carefully prepared dataset gives the model a fair opportunity to learn useful patterns.